In [23]:
#import librosa
#import IPython.display as ipd
import requests
import io
import re

from google.cloud import storage
import os

#import soundfile as sf

# Note
Mixer functions originally built for wav files but some of these are stored as flac / other types - may need to change mixer functions to sf.read() instead of wavfile.read()

# Step 1
Get URLs from event buckets of interest and store them in a dictionary that has a user-friendly key for the clip.
I need to expand this code to iterate through all folders with sound clips and not just this one but this is the idea:

In [22]:
bucket_name = "noaa-passive-bioacoustic"
prefix = "sanctsound/products/sound_clips/"

client = storage.Client.create_anonymous_client()
bucket = client.bucket(bucket_name)
blobs = client.list_blobs(bucket, prefix=prefix)

urls = {}

for blob in blobs:
    if not blob.name.lower().endswith(".wav"):
        continue

    filename = blob.name.split("/")[-1]

    # SanctSound_MB01_05_bocaccio_20200508T042435Z.wav
    match = re.match(
        r"SanctSound_([A-Za-z0-9]+_[0-9]+)_([^_]+)_\d{8}T\d{6}Z\.wav$",
        filename,
        re.IGNORECASE
    )

    if not match:
        continue

    site_code = match.group(1).upper()   # MB01_05
    event_name = match.group(2).lower()  # bocaccio

    # Format species name
    if "whale" in event_name:
        parts = event_name.split("whale")
        species = " ".join([p.capitalize() for p in parts if p] + ["Whale"])
    else:
        species = event_name.capitalize()

    key = f"{species} - NOAA SanctSound {site_code}"

    urls.setdefault(key, []).append(
        f"gs://{bucket_name}/{blob.name}"
    )

Bocaccio - NOAA SanctSound CI01_01
Snappingshrimp - NOAA SanctSound CI01_01
Soundscape - NOAA SanctSound CI01_01
Plainfinmidshipman - NOAA SanctSound CI01_02
Dolphins - NOAA SanctSound CI01_03
Smallboat - NOAA SanctSound CI01_03
Snappingshrimp - NOAA SanctSound CI01_03
Soundscape - NOAA SanctSound CI01_03
Odontocetewhistlesandbuzzes - NOAA SanctSound CI01_05
Odontocetewhistlesandclicks - NOAA SanctSound CI01_05
Plainfinmidshipman - NOAA SanctSound CI01_05
Snappingshrimp - NOAA SanctSound CI01_05
Soundscape - NOAA SanctSound CI01_05
Fin Whale - NOAA SanctSound CI02_01
Soundscape - NOAA SanctSound CI02_01
Wind - NOAA SanctSound CI02_03
Humpback Whale - NOAA SanctSound CI02_04
Humpback Song Whale - NOAA SanctSound CI02_04
Soundscape - NOAA SanctSound CI02_04
Unknownfishknocks - NOAA SanctSound CI02_04
Humpback Whale - NOAA SanctSound CI02_05
Humpback Song Whale - NOAA SanctSound CI02_05
Odontoceteclicksandwhistle - NOAA SanctSound CI02_05
Odontocetewhistles - NOAA SanctSound CI02_05
Seali

In [24]:

    
keys_to_keep = {
    "Bocaccio - NOAA SanctSound CI01_01"#,
    "Snappingshrimp - NOAA SanctSound CI01_01",
    "Fin Whale - NOAA SanctSound CI02_01",
    "Humpback Whale - NOAA SanctSound CI02_04",
    "Wind - NOAA SanctSound CI02_03",
    "Humpback Whale - NOAA SanctSound CI02_05",
    "Sealionbark - NOAA SanctSound CI02_05",
    "Ship - NOAA SanctSound CI02_05",
    "Vessel - NOAA SanctSound CI04_01",
    "Humpback Song Whale - NOAA SanctSound CI04_04",
    "Dolphins - NOAA SanctSound CI04_05",
    "Lfasonar - NOAA SanctSound CI05_01",
    "Sealbomb - NOAA SanctSound CI05_02",
    "Blue Whale - NOAA SanctSound CI05_03",
    "Largeship - NOAA SanctSound CI05_03",
    "Fin Whale - NOAA SanctSound CI05_04",
    "Blackgrouper - NOAA SanctSound FK01_01",
    "Scubadivers - NOAA SanctSound FK01_06",
    "Windwaves - NOAA SanctSound FK02_03",
    "Hurricane - NOAA SanctSound FK02_05",
    "Redgrouper - NOAA SanctSound FK03_01",
    "Northatlanticright Whale - NOAA SanctSound GR01_01",
    "Toadfish - NOAA SanctSound GR01_01",
    "Fishchorus - NOAA SanctSound GR01_02",
    "Rain - NOAA SanctSound GR01_02",
    "Hurricane - NOAA SanctSound GR03_02",
    "Minke Whale - NOAA SanctSound HI03_02",
    "Sperm Whale - NOAA SanctSound HI03_02",
    "Sonar - NOAA SanctSound HI04_02",
    "Blue Bcall Whale - NOAA SanctSound MB01_04",
    "Blue Acall Whale - NOAA SanctSound MB01_04",
    "Cruiseship - NOAA SanctSound MB02_03",
    "Killer Whale - NOAA SanctSound OC03_02",
    "Damselfish - NOAA SanctSound PM05_01",
    "Minke Whale - NOAA SanctSound PM05_01"
}

urls_to_save = {k: v[0] for k, v in urls.items()}

urls_to_save

{'Bocaccio - NOAA SanctSound CI01_01': 'gs://noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_01_sound_clips/data/SanctSound_CI01_01_bocaccio_20181101T100353Z.wav',
 'Snappingshrimp - NOAA SanctSound CI01_01': 'gs://noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_01_sound_clips/data/SanctSound_CI01_01_snappingshrimp_20181101T080448Z.wav',
 'Soundscape - NOAA SanctSound CI01_01': 'gs://noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_01_sound_clips/data/SanctSound_CI01_01_soundscape_20181106T132459Z.wav',
 'Plainfinmidshipman - NOAA SanctSound CI01_02': 'gs://noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_02_sound_clips/data/SanctSound_CI01_02_plainfinmidshipman_20190426T085831Z.wav',
 'Dolphins - NOAA SanctSound CI01_03': 'gs://noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_03_sound_clips/data/SanctSound_CI01_03_dolphins_20190904T064203Z.

limit_output extension: Maximum message size of 10000 exceeded with 40667 characters

In [25]:
urls_dict = urls_to_save  # replace with your actual dictionary

# Create downloads folder
os.makedirs("downloads", exist_ok=True)

# Create anonymous client for public GCS
client = storage.Client.create_anonymous_client()

for key, file_url in urls_dict.items():
    # Make a safe filename from the key
    safe_filename = key + ".wav"
    local_path = os.path.join("downloads", safe_filename)
    
    # Parse bucket and blob path from gs:// URL
    _, path = file_url.split("gs://", 1)
    bucket_name, blob_path = path.split("/", 1)
    
    # Download
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    blob.download_to_filename(local_path)

# Step 2
Function that will load audio for a given key.
My thought is user could "add" individual files they want to use to their library since it would probably take too long to store all in local memory (even though they'd be stored numerically)

In [66]:
def load_audio(key):

    if key not in urls:
        raise(ValueError(f"Key not found."))

    full_url = f"https://storage.googleapis.com/{urls[key]}"
    response = requests.get(full_url)
    response.raise_for_status()  # raise error if request failed
    audio_bytes = io.BytesIO(response.content)
    
    # Read audio
    data, sr = sf.read(audio_bytes)
    
    return data, sr

load_audio("Killer whale - SanctSound")

(array([ 0.00000000e+00,  3.05175781e-05, -6.10351562e-05, ...,
        -1.83105469e-04,  4.27246094e-04, -3.35693359e-04],
       shape=(2880000,)),
 48000)